
# SFTP → Microsoft Fabric Lakehouse: Hybrid Copy (Parallel + Structured Logs + Bandwidth Limiter + Key Vault)

This notebook ingests files from an **SFTP** server into **Microsoft Fabric Lakehouse** and preserves the
remote directory hierarchy under your chosen Lakehouse `/Files` folder.

### Key capabilities
- **Auth**: Password or SSH private key (optionally pulled from **Azure Key Vault**)
- **Recursive sync**: Walks subfolders and mirrors the hierarchy
- **Filter**: `include_exts` (e.g., `[".csv", ".parquet"]`)
- **Hybrid copy**: Buffered for small files, **streaming** for large files (size threshold + configurable chunk size)
- **Parallelism**: Thread-per-connection workers with retries + backoff
- **Structured logging**: JSON or text, with `run_id` correlation and optional rotating file persisted to Lakehouse
- **Global bandwidth limiter**: Token-bucket, shared across all workers (caps total throughput)
- **Dry run**: Preview actions without copying
- **Verification** (optional): Quick count of mirrored files in Lakehouse

### How to use in Microsoft Fabric
1. **Upload**: Fabric workspace → *Notebook* → **Upload** → select this file `SFTP_to_Lakehouse_Hybrid.ipynb`.
2. **Attach a Lakehouse**: Click **Add Lakehouse** so paths like `/lakehouse/default/Files/...` work.
3. **Edit the CONFIG** (Cell 4):
   - `host`, `port`, `username`, and either `password` or `key_path` (or enable **Key Vault** to populate them)
   - `sftp_root` (remote root to mirror) and `lakehouse_base` (destination under Lakehouse `/Files`)
   - `include_exts`
   - Hybrid knobs: `large_file_threshold_mb`, `chunk_size_mb`
   - Parallel knobs: `max_workers`, `retries`, `retry_backoff_base`
   - Logging knobs: `log_format`, `log_to_file`, `log_dir`, etc.
   - Limiter knobs: `bandwidth_limit_mbps`, `burst_capacity_seconds`, `force_streaming_under_limiter`
   - Key Vault knobs: connection name or vault URL + secret names
   - Start with `dry_run=True` to preview
4. **Run cells in order**. The Run cell prints a summary and logs details to output and (optionally) to `/lakehouse/.../Files/logs/`.


In [ ]:

# Install required packages
!pip install --quiet paramiko azure-identity azure-keyvault-secrets


In [ ]:

# === HELPERS: Structured Logging, Global Bandwidth Limiter, SFTP Parallel Copy, Key Vault ===
import os
import time
import threading
import stat
import json
import logging
from logging.handlers import RotatingFileHandler
from typing import List, Tuple, Dict, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import paramiko

# -------------------------
# Structured logging setup
# -------------------------
class JsonFormatter(logging.Formatter):
    def __init__(self, *, default_extra: Optional[Dict] = None):
        super().__init__()
        self.default_extra = default_extra or {}

    def format(self, record: logging.LogRecord) -> str:
        payload = {
            "ts": self.formatTime(record, datefmt="%Y-%m-%dT%H:%M:%S.%fZ"),
            "level": record.levelname,
            "logger": record.name,
            "thread": record.threadName,
            "msg": record.getMessage(),
        }
        if record.levelno <= logging.DEBUG:
            payload["module"] = record.module
            payload["func"] = record.funcName
            payload["lineno"] = record.lineno
        reserved = set(vars(logging.makeLogRecord({})).keys())
        for k, v in record.__dict__.items():
            if k not in reserved and k not in payload:
                try:
                    json.dumps(v)
                    payload[k] = v
                except Exception:
                    payload[k] = str(v)
        for k, v in self.default_extra.items():
            payload.setdefault(k, v)
        return json.dumps(payload, ensure_ascii=False)

def setup_logging(
    level: str = "INFO",
    fmt: str = "json",
    to_file: bool = False,
    file_path: Optional[str] = None,
    max_bytes: int = 50 * 1024 * 1024,
    backup_count: int = 2,
    default_extra: Optional[Dict] = None
) -> logging.LoggerAdapter:
    logger = logging.getLogger("sftp_hybrid")
    logger.handlers.clear()
    logger.propagate = False
    logger.setLevel(getattr(logging, level.upper(), logging.INFO))

    if fmt.lower() == "json":
        formatter = JsonFormatter(default_extra=default_extra)
    else:
        fmt_str = "%(asctime)s %(levelname)s [%(name)s] [%(threadName)s] %(message)s"
        formatter = logging.Formatter(fmt_str, datefmt="%Y-%m-%dT%H:%M:%S")

    stream = logging.StreamHandler()
    stream.setFormatter(formatter)
    logger.addHandler(stream)

    if to_file and file_path:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        fh = RotatingFileHandler(file_path, maxBytes=max_bytes, backupCount=backup_count)
        fh.setFormatter(formatter)
        logger.addHandler(fh)

    return logging.LoggerAdapter(logger, default_extra or {})

# -------------------------
# Global Token-Bucket Limiter
# -------------------------
class GlobalTokenBucketLimiter:
    """Thread-safe global token-bucket limiter shared across workers."""
    def __init__(self, rate_bytes_per_sec: int, capacity_bytes: Optional[int] = None,
                 min_sleep_sec: float = 0.01, logger: Optional[logging.LoggerAdapter] = None):
        self.enabled = rate_bytes_per_sec is not None and rate_bytes_per_sec > 0
        self.rate = float(rate_bytes_per_sec) if self.enabled else 0.0
        self.capacity = int(capacity_bytes) if (self.enabled and capacity_bytes) else int(self.rate * 2) if self.enabled else 0
        self.tokens = self.capacity
        self.last = time.monotonic()
        self.lock = threading.Lock()
        self.min_sleep = float(min_sleep_sec)
        self.logger = logger
        if self.enabled and self.logger:
            self.logger.info("Limiter initialized", extra={
                "event": "limiter_init",
                "rate_bytes_per_sec": int(self.rate),
                "capacity_bytes": self.capacity
            })

    def _refill(self, now: float):
        elapsed = now - self.last
        if elapsed <= 0:
            return
        add = elapsed * self.rate
        if add > 0:
            self.tokens = min(self.capacity, self.tokens + add)
            self.last = now

    def acquire(self, n: int):
        if not self.enabled or n <= 0:
            return
        waited = 0.0
        while True:
            with self.lock:
                now = time.monotonic()
                self._refill(now)
                if self.tokens >= n:
                    self.tokens -= n
                    if waited > 0 and self.logger:
                        self.logger.debug("Limiter waited", extra={
                            "event": "limiter_wait",
                            "bytes": n,
                            "waited_sec": round(waited, 4)
                        })
                    return
                deficit = n - self.tokens
                sleep_sec = max(self.min_sleep, deficit / self.rate if self.rate > 0 else self.min_sleep)
            time.sleep(sleep_sec)
            waited += sleep_sec

# -------------------------
# SFTP connection helpers
# -------------------------
_thread_local = threading.local()

def _load_pkey_from_file(path: str, passphrase: Optional[str] = None):
    last_err = None
    for cls in (paramiko.RSAKey, getattr(paramiko, 'Ed25519Key', None), paramiko.ECDSAKey, paramiko.DSSKey):
        if cls is None:
            continue
        try:
            return cls.from_private_key_file(path, password=passphrase)
        except Exception as e:
            last_err = e
            continue
    raise last_err if last_err else RuntimeError("Unsupported key type or unreadable key file")

def _make_sftp_client(host: str, port: int, username: str,
                      password: Optional[str] = None, key_path: Optional[str] = None,
                      key_passphrase: Optional[str] = None,
                      socket_timeout_sec: int = 60, keepalive_secs: int = 30,
                      logger: Optional[logging.LoggerAdapter] = None) -> paramiko.SFTPClient:
    transport = None
    try:
        transport = paramiko.Transport((host, port))
        if key_path:
            pkey = _load_pkey_from_file(key_path, passphrase=key_passphrase)
            transport.connect(username=username, pkey=pkey)
        else:
            transport.connect(username=username, password=password)
        transport.set_keepalive(keepalive_secs)
        sftp = paramiko.SFTPClient.from_transport(transport)
        try:
            sftp.get_channel().settimeout(socket_timeout_sec)
        except Exception:
            pass
        if logger:
            logger.debug("SFTP client created", extra={"event": "sftp_connect"})
        return sftp
    except Exception as e:
        if logger:
            logger.error("SFTP connect failed", extra={"event": "sftp_connect_failed", "error": str(e)})
        if transport:
            try:
                transport.close()
            except Exception:
                pass
        raise

def _get_thread_sftp(**conn_kwargs) -> paramiko.SFTPClient:
    sftp = getattr(_thread_local, "sftp", None)
    if sftp is None:
        sftp = _make_sftp_client(**conn_kwargs)
        _thread_local.sftp = sftp
    return sftp

def _close_thread_sftp():
    sftp = getattr(_thread_local, "sftp", None)
    if sftp is not None:
        try:
            try:
                sftp.close()
            except Exception:
                pass
            try:
                chan = sftp.get_channel()
                if chan:
                    tr = chan.get_transport()
                    if tr:
                        tr.close()
            except Exception:
                pass
        finally:
            _thread_local.sftp = None

def _ext_ok(path: str, include_exts: List[str]) -> bool:
    if not include_exts:
        return True
    lower = path.lower()
    return any(lower.endswith(ext.lower()) for ext in include_exts)

def _ensure_dir(path: str):
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)

# -------------------------
# Key Vault helpers
# -------------------------
# Try to import Fabric/Synapse mssparkutils for secrets via connection
_mssu = None
try:
    from notebookutils import mssparkutils as _mssu
except Exception:
    try:
        import mssparkutils as _mssu
    except Exception:
        _mssu = None

# Lazy imports for Azure SDK
_AZURE_SDK_AVAILABLE = True
try:
    from azure.identity import DefaultAzureCredential
    from azure.keyvault.secrets import SecretClient
except Exception:
    _AZURE_SDK_AVAILABLE = False


def kv_get_secret_fabric(connection_name: str, secret_name: str, logger: Optional[logging.LoggerAdapter] = None) -> Optional[str]:
    if _mssu is None:
        raise RuntimeError("mssparkutils not available. Ensure you're running in a Fabric/Spark notebook and the Key Vault connection exists.")
    if logger:
        logger.debug("Getting secret via Fabric connection", extra={"event": "kv_fetch", "mode": "fabric_connection", "connection": connection_name, "secret": secret_name})
    return _mssu.credentials.getSecret(connection_name, secret_name)


def kv_get_secret_sdk(vault_url: str, secret_name: str, logger: Optional[logging.LoggerAdapter] = None) -> Optional[str]:
    if not _AZURE_SDK_AVAILABLE:
        raise RuntimeError("Azure SDK not available. Install azure-identity and azure-keyvault-secrets.")
    if logger:
        logger.debug("Getting secret via SDK", extra={"event": "kv_fetch", "mode": "sdk", "vault_url": vault_url, "secret": secret_name})
    cred = DefaultAzureCredential()
    client = SecretClient(vault_url=vault_url, credential=cred)
    return client.get_secret(secret_name).value


def kv_resolve_sftp_credentials(
    use_key_vault: bool,
    kv_mode: str,
    kv_connection_name: Optional[str],
    kv_vault_url: Optional[str],
    kv_secret_password_name: Optional[str],
    kv_secret_pem_name: Optional[str],
    kv_secret_pem_passphrase_name: Optional[str],
    key_write_path: str,
    logger: Optional[logging.LoggerAdapter] = None
) -> Dict[str, Optional[str]]:
    """
    Returns dict with possible keys: password, key_path, key_passphrase.
    - If a PEM secret is provided, writes it to key_write_path and sets chmod 600.
    - Does not log secret contents.
    """
    result = {"password": None, "key_path": None, "key_passphrase": None}
    if not use_key_vault:
        return result

    def _get(name: Optional[str]) -> Optional[str]:
        if not name:
            return None
        if kv_mode == "fabric_connection":
            return kv_get_secret_fabric(kv_connection_name, name, logger)
        elif kv_mode == "sdk":
            return kv_get_secret_sdk(kv_vault_url, name, logger)
        else:
            raise ValueError("kv_mode must be 'fabric_connection' or 'sdk'")

    # Fetch password
    pwd = _get(kv_secret_password_name)
    if pwd:
        result["password"] = pwd
        if logger:
            logger.info("Got SFTP password from Key Vault", extra={"event": "kv_password_ok"})

    # Fetch PEM and optional passphrase
    pem = _get(kv_secret_pem_name)
    if pem:
        try:
            _ensure_dir(key_write_path)
            with open(key_write_path, "w", encoding="utf-8") as f:
                f.write(pem)
            try:
                os.chmod(key_write_path, 0o600)
            except Exception:
                pass
            result["key_path"] = key_write_path
            if logger:
                logger.info("Wrote private key PEM from Key Vault", extra={"event": "kv_pem_ok", "key_path": key_write_path})
        except Exception as e:
            if logger:
                logger.error("Failed to write PEM to disk", extra={"event": "kv_pem_write_failed", "error": str(e)})
            raise

        # Optional passphrase
        pphrase = _get(kv_secret_pem_passphrase_name)
        if pphrase:
            result["key_passphrase"] = pphrase
            if logger:
                logger.info("Loaded PEM passphrase from Key Vault", extra={"event": "kv_pem_passphrase_ok"})

    return result

# -------------------------
# Discovery (single-thread)
# -------------------------

def discover_remote_files(host: str, port: int, username: str, password: Optional[str], key_path: Optional[str],
                          sftp_root: str, include_exts: List[str],
                          socket_timeout_sec: int, keepalive_secs: int,
                          logger: logging.LoggerAdapter) -> List[Tuple[str, str, int]]:
    t0 = time.time()
    logger.info("Starting discovery", extra={"event": "discovery_start", "sftp_root": sftp_root})
    sftp = _make_sftp_client(host=host, port=port, username=username, password=password, key_path=key_path,
                             key_passphrase=None, socket_timeout_sec=socket_timeout_sec, keepalive_secs=keepalive_secs, logger=logger)
    tasks = []
    dirs_scanned = 0
    try:
        root = sftp_root.rstrip("/")
        stack = [root]
        while stack:
            current = stack.pop()
            try:
                entries = sftp.listdir_attr(current)
                dirs_scanned += 1
            except Exception as e:
                logger.warning("Failed to list directory", extra={"event": "discovery_list_failed", "path": current, "error": str(e)})
                continue
            for entry in entries:
                name = entry.filename
                full_path = f"{current}/{name}" if not current.endswith("/") else f"{current}{name}"
                if stat.S_ISDIR(entry.st_mode):
                    stack.append(full_path)
                else:
                    rel = full_path[len(root):].lstrip("/")
                    if _ext_ok(full_path, include_exts):
                        tasks.append((full_path, rel, entry.st_size))
    finally:
        try:
            sftp.close()
        except Exception:
            pass
        try:
            ch = sftp.get_channel()
            if ch:
                tr = ch.get_transport()
                if tr:
                    tr.close()
        except Exception:
            pass
    elapsed = round(time.time() - t0, 2)
    logger.info("Discovery complete", extra={
        "event": "discovery_complete",
        "dirs_scanned": dirs_scanned,
        "files_seen": len(tasks),
        "elapsed_sec": elapsed
    })
    return tasks

# -------------------------
# Copy primitives (limiter-aware)
# -------------------------

def _copy_streaming_with_limiter(sftp: paramiko.SFTPClient, remote_path: str, local_path: str,
                                 chunk_size_bytes: int, limiter: Optional[GlobalTokenBucketLimiter]):
    _ensure_dir(local_path)
    with sftp.open(remote_path, "rb") as rf, open(local_path, "wb") as lf:
        while True:
            chunk = rf.read(chunk_size_bytes)
            if not chunk:
                break
            if limiter and limiter.enabled:
                limiter.acquire(len(chunk))
            lf.write(chunk)

def _copy_small_buffered(sftp: paramiko.SFTPClient, remote_path: str, local_path: str):
    with sftp.open(remote_path, "rb") as rf:
        data = rf.read()
    _ensure_dir(local_path)
    with open(local_path, "wb") as lf:
        lf.write(data)

# -------------------------
# Worker task
# -------------------------

def copy_one_file_task(task: Tuple[str, str, int],
                       lakehouse_base: str,
                       large_file_threshold_mb: int,
                       chunk_size_mb: int,
                       skip_if_same_size: bool,
                       dry_run: bool,
                       conn_kwargs: Dict,
                       retries: int,
                       retry_backoff_base: float,
                       create_missing_dirs: bool,
                       logger: logging.LoggerAdapter,
                       limiter: Optional[GlobalTokenBucketLimiter],
                       force_streaming_under_limiter: bool) -> Dict:
    remote_path, rel, remote_size = task
    local_path = os.path.join(lakehouse_base.rstrip("/"), rel)

    threshold_bytes = int(large_file_threshold_mb * 1024 * 1024)
    chunk_size_bytes = max(1, int(chunk_size_mb * 1024 * 1024))

    if limiter and limiter.enabled and force_streaming_under_limiter:
        predicted_mode = "streaming"
    else:
        predicted_mode = "streaming" if remote_size >= threshold_bytes else "buffered"

    if dry_run:
        logger.info("Dry-run: would copy", extra={
            "event": "copy_dry_run",
            "remote_path": remote_path,
            "local_path": local_path,
            "bytes": remote_size,
            "mode": predicted_mode
        })
        return {"status": "dry_run", "remote_path": remote_path, "local_path": local_path,
                "bytes": remote_size, "mode": "n/a", "attempts": 0}

    if skip_if_same_size and os.path.exists(local_path):
        try:
            if os.path.getsize(local_path) == remote_size:
                logger.info("Skip (same size)", extra={
                    "event": "copy_skip_same_size",
                    "remote_path": remote_path,
                    "local_path": local_path,
                    "bytes": remote_size
                })
                return {"status": "skipped_same_size", "remote_path": remote_path, "local_path": local_path,
                        "bytes": remote_size, "mode": "n/a", "attempts": 0}
        except Exception as e:
            logger.debug("Local size check failed; will attempt copy", extra={
                "event": "local_stat_error",
                "local_path": local_path,
                "error": str(e)
            })

    attempts = 0
    last_err = None
    t0 = time.time()

    while attempts <= retries:
        attempts += 1
        try:
            sftp = _get_thread_sftp(**conn_kwargs)
            if create_missing_dirs:
                _ensure_dir(local_path)
            logger.debug("Copy start", extra={
                "event": "copy_start",
                "remote_path": remote_path,
                "local_path": local_path,
                "bytes": remote_size,
                "attempt": attempts,
                "mode": predicted_mode
            })
            if predicted_mode == "streaming":
                _copy_streaming_with_limiter(sftp, remote_path, local_path, chunk_size_bytes, limiter)
                mode = "streaming"
            else:
                _copy_small_buffered(sftp, remote_path, local_path)
                mode = "buffered"
            dur = round(time.time() - t0, 3)
            logger.info("Copy complete", extra={
                "event": "copy_done",
                "remote_path": remote_path,
                "local_path": local_path,
                "bytes": remote_size,
                "attempts": attempts,
                "mode": mode,
                "duration_sec": dur
            })
            return {"status": "copied", "remote_path": remote_path, "local_path": local_path,
                    "bytes": remote_size, "mode": mode, "attempts": attempts}
        except Exception as e:
            last_err = e
            try:
                _close_thread_sftp()
            except Exception:
                pass
            if attempts <= retries:
                sleep_s = retry_backoff_base ** attempts
                logger.warning("Copy failed; will retry", extra={
                    "event": "copy_retry",
                    "remote_path": remote_path,
                    "local_path": local_path,
                    "bytes": remote_size,
                    "attempt": attempts,
                    "retries_remaining": max(0, retries - attempts + 1),
                    "retry_sleep_sec": round(sleep_s, 3),
                    "error": str(e)
                })
                time.sleep(sleep_s)
            else:
                break

    logger.error("Copy failed (exhausted retries)", extra={
        "event": "copy_failed",
        "remote_path": remote_path,
        "local_path": local_path,
        "bytes": remote_size,
        "attempts": attempts,
        "error": f"{type(last_err).__name__}: {last_err}"
    })
    return {"status": "failed", "remote_path": remote_path, "local_path": local_path,
            "bytes": remote_size, "mode": "n/a", "attempts": attempts,
            "error": f"{type(last_err).__name__}: {last_err}"}

# -------------------------
# Orchestrator
# -------------------------

def run_parallel_copy(host: str, port: int, username: str, password: Optional[str], key_path: Optional[str],
                      key_passphrase: Optional[str],
                      sftp_root: str, lakehouse_base: str, include_exts: List[str],
                      large_file_threshold_mb: int, chunk_size_mb: int,
                      max_workers: int, retries: int, retry_backoff_base: float,
                      skip_if_same_size: bool, dry_run: bool, create_missing_dirs: bool,
                      socket_timeout_sec: int, keepalive_secs: int,
                      logger: logging.LoggerAdapter,
                      bandwidth_limit_mbps: Optional[float], burst_capacity_seconds: float,
                      min_sleep_ms: int, force_streaming_under_limiter: bool) -> Dict:
    t0 = time.time()
    logger.info("Run start", extra={
        "event": "run_start",
        "sftp_root": sftp_root,
        "lakehouse_base": lakehouse_base,
        "max_workers": max_workers,
        "dry_run": dry_run
    })

    tasks = discover_remote_files(host, port, username, password, key_path, sftp_root, include_exts,
                                  socket_timeout_sec, keepalive_secs, logger)

    # Build global limiter
    limiter = None
    if bandwidth_limit_mbps and bandwidth_limit_mbps > 0:
        rate_bps = float(bandwidth_limit_mbps) * 125_000.0  # Mbps -> bytes/sec
        capacity = int(rate_bps * max(0.1, float(burst_capacity_seconds)))
        limiter = GlobalTokenBucketLimiter(rate_bytes_per_sec=int(rate_bps), capacity_bytes=capacity,
                                           min_sleep_sec=max(0.001, float(min_sleep_ms) / 1000.0), logger=logger)
        logger.info("Limiter active", extra={
            "event": "limiter_active",
            "bandwidth_limit_mbps": bandwidth_limit_mbps,
            "capacity_bytes": capacity
        })
    else:
        logger.info("Limiter disabled", extra={"event": "limiter_disabled"})

    if dry_run and not tasks:
        elapsed = round(time.time() - t0, 2)
        logger.info("Dry-run: nothing to do", extra={"event": "dry_run_noop", "elapsed_sec": elapsed})
        return {"files_seen": 0, "files_copied": 0, "files_skipped": 0, "files_failed": 0,
                "streaming_ops": 0, "elapsed_sec": elapsed}

    conn_kwargs = dict(host=host, port=port, username=username, password=password, key_path=key_path,
                       key_passphrase=key_passphrase,
                       socket_timeout_sec=socket_timeout_sec, keepalive_secs=keepalive_secs, logger=logger)

    copied = skipped = failed = streaming_ops = 0

    if dry_run:
        for remote_path, rel, size in tasks:
            local_path = os.path.join(lakehouse_base.rstrip("/"), rel)
            if skip_if_same_size and os.path.exists(local_path) and os.path.getsize(local_path) == size:
                skipped += 1
            else:
                copied += 1
        elapsed = round(time.time() - t0, 2)
        logger.info("Dry-run summary", extra={
            "event": "dry_run_summary",
            "files_seen": len(tasks),
            "would_copy": copied,
            "would_skip": skipped,
            "elapsed_sec": elapsed
        })
        return {"files_seen": len(tasks), "files_copied": copied, "files_skipped": skipped,
                "files_failed": failed, "streaming_ops": streaming_ops, "elapsed_sec": elapsed}

    logger.info("Starting parallel copy", extra={"event": "copy_start_all", "workers": max_workers, "queue": len(tasks)})

    try:
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = [
                pool.submit(copy_one_file_task, t, lakehouse_base, large_file_threshold_mb, chunk_size_mb,
                            skip_if_same_size, dry_run, conn_kwargs, retries, retry_backoff_base,
                            create_missing_dirs, logger, limiter, force_streaming_under_limiter)
                for t in tasks
            ]
            for fut in as_completed(futures):
                res = fut.result()
                st = res.get("status")
                if st == "copied":
                    copied += 1
                    if res.get("mode") == "streaming":
                        streaming_ops += 1
                elif st.startswith("skipped"):
                    skipped += 1
                elif st == "dry_run":
                    skipped += 1
                else:
                    failed += 1
    except KeyboardInterrupt:
        logger.warning("Cancellation requested", extra={"event": "run_cancel"})
    finally:
        try:
            _close_thread_sftp()
        except Exception:
            pass

    elapsed = round(time.time() - t0, 2)
    logger.info("Run summary", extra={
        "event": "run_summary",
        "files_seen": len(tasks),
        "files_copied": copied,
        "files_skipped": skipped,
        "files_failed": failed,
        "streaming_ops": streaming_ops,
        "elapsed_sec": elapsed
    })

    return {"files_seen": len(tasks), "files_copied": copied, "files_skipped": skipped,
            "files_failed": failed, "streaming_ops": streaming_ops, "elapsed_sec": elapsed}


In [ ]:

# === CONFIG ===
host = "your.sftp.server"
port = 22
username = "user"
password = None              # If using Key Vault, leave None
key_path = None              # If using Key Vault, leave None
key_passphrase = None        # Optional passphrase for encrypted private key

sftp_root = "/remote/source/root"  # remote path to mirror
lakehouse_base = "/lakehouse/default/Files/sftp_sync"  # destination in Lakehouse Files

# Filter by extension (case-insensitive)
include_exts = [".csv"]

# Hybrid copy knobs
large_file_threshold_mb = 1024   # 1 GB
chunk_size_mb = 8                # streaming chunk size in MB

# Parallelism + resiliency
max_workers = 8                  # tune based on SFTP server limits
retries = 3                      # retries per file
retry_backoff_base = 1.5         # exponential backoff factor

# Safety & convenience
dry_run = True                   # start with True to preview
skip_if_same_size = True         # idempotent skip by size match
create_missing_dirs = True       # create local dirs in Lakehouse destination

# Network
socket_timeout_sec = 60
keepalive_secs = 30

# ---- Structured logging ----
log_level = "INFO"              # DEBUG | INFO | WARNING | ERROR
log_format = "json"             # "json" or "text"
log_to_file = True              # Persist logs to Lakehouse files
log_dir = "/lakehouse/default/Files/logs"
log_max_bytes = 50 * 1024 * 1024
log_backup_count = 2

# ---- Global bandwidth limiter ----
# Set to 0 or None to disable
bandwidth_limit_mbps = 100       # e.g., 100 Mbps cap
burst_capacity_seconds = 2.0     # bucket size as seconds of allowance
min_sleep_ms = 10                # minimum sleep quantum during throttling
force_streaming_under_limiter = True  # stream even small files when limiting

# ---- Azure Key Vault (optional) ----
use_key_vault = True
kv_mode = "fabric_connection"       # "fabric_connection" or "sdk"
kv_connection_name = "kv-connection-name"   # Fabric connection name (if kv_mode="fabric_connection")
kv_vault_url = "https://<your-vault>.vault.azure.net/"  # Required if kv_mode="sdk"

# Secret names to fetch (set whichever you use)
kv_secret_password_name = None                # e.g., "sftp-password"
kv_secret_pem_name = None                     # e.g., "sftp-privatekey-pem" (PEM content)
kv_secret_pem_passphrase_name = None          # e.g., "sftp-privatekey-passphrase" (optional)
kv_write_key_to = f"/tmp/sftp_kv_key_{'${'}run_id{'}'}.pem"   # path to write PEM (ephemeral)

# Per-run correlation id & log path
import uuid, datetime, os
run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + str(uuid.uuid4())[:8]
os.makedirs(log_dir, exist_ok=True)
log_file_path = f"{log_dir}/sftp_hybrid_{run_id}.log"
print("CONFIG loaded. run_id =", run_id)

# --- initialize logger ---
logger = setup_logging(
    level=log_level,
    fmt=log_format,
    to_file=log_to_file,
    file_path=log_file_path,
    max_bytes=log_max_bytes,
    backup_count=log_backup_count,
    default_extra={"run_id": run_id, "host": host}
)
logger.info("Logging initialized", extra={"event": "logging_init", "log_file": log_file_path if log_to_file else None})

# --- Resolve secrets from Key Vault (optional) ---
if use_key_vault:
    try:
        resolved = kv_resolve_sftp_credentials(
            use_key_vault=use_key_vault,
            kv_mode=kv_mode,
            kv_connection_name=kv_connection_name,
            kv_vault_url=kv_vault_url,
            kv_secret_password_name=kv_secret_password_name,
            kv_secret_pem_name=kv_secret_pem_name,
            kv_secret_pem_passphrase_name=kv_secret_pem_passphrase_name,
            key_write_path=kv_write_key_to,
            logger=logger
        )
        # Override from KV if present
        if resolved.get("password"):
            password = resolved["password"]
        if resolved.get("key_path"):
            key_path = resolved["key_path"]
        if resolved.get("key_passphrase"):
            key_passphrase = resolved["key_passphrase"]
    except Exception as e:
        logger.error("Key Vault resolution failed", extra={"event": "kv_failed", "error": str(e)})
        raise

# --- run orchestrator ---
summary = run_parallel_copy(
    host=host, port=port, username=username, password=password, key_path=key_path, key_passphrase=key_passphrase,
    sftp_root=sftp_root, lakehouse_base=lakehouse_base, include_exts=include_exts,
    large_file_threshold_mb=large_file_threshold_mb, chunk_size_mb=chunk_size_mb,
    max_workers=max_workers, retries=retries, retry_backoff_base=retry_backoff_base,
    skip_if_same_size=skip_if_same_size, dry_run=dry_run, create_missing_dirs=create_missing_dirs,
    socket_timeout_sec=socket_timeout_sec, keepalive_secs=keepalive_secs,
    logger=logger,
    bandwidth_limit_mbps=bandwidth_limit_mbps,
    burst_capacity_seconds=burst_capacity_seconds,
    min_sleep_ms=min_sleep_ms,
    force_streaming_under_limiter=force_streaming_under_limiter
)
print("Summary:", summary)


In [ ]:

# Optional: quick verification to count mirrored files in Lakehouse
import os

def count_files_with_ext(root: str, include_exts):
    total = 0
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if not include_exts:
                total += 1
            else:
                lf = f.lower()
                if any(lf.endswith(ext.lower()) for ext in include_exts):
                    total += 1
    return total

count = count_files_with_ext(lakehouse_base, include_exts)
print(f"Lakehouse folder: {lakehouse_base}")
print(f"Files matching {include_exts}: {count}")
